# F-Smart Memories: pretrained quickstart cho Kaggle

Notebook khởi động: **tải checkpoint AdaFace chính chủ** và **chạy Fast Neural Style Transfer của PyTorch**. Không train model. Notebook này chưa chứa pipeline tìm kiếm khuôn mặt hoặc full app.

1. Bật Internet; chọn GPU nếu session có GPU.
2. Chạy cell môi trường, cell tải AdaFace và các cell NST.
3. Sửa `CONTENT_IMAGE` thành đường dẫn ảnh của bạn ở cell demo.

Phần tải AdaFace chỉ tải snapshot và kiểm tra file, chưa load encoder. Inference AdaFace cần loader/alignment đầy đủ theo model card. Chi tiết quyết định model và dữ liệu nằm trong `docs/F_SMART_MEMORIES_PLAN.vi.md`.

Nguồn: [AdaFace/CVLFace HF](https://huggingface.co/minchul/cvlface_adaface_ir50_webface4m), [PyTorch Fast Neural Style](https://github.com/pytorch/examples/tree/main/fast_neural_style).

**Trạng thái kiểm chứng:** JSON và cú pháp Python đã được kiểm tra tại máy tạo notebook. Chưa chạy notebook trên Kaggle, chưa tải/kiểm tra toàn bộ checkpoint hay đo chất lượng đầu ra. Giữ notebook/dataset ảnh cá nhân ở chế độ riêng tư.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import importlib.util
import json
import platform
import re
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile

import numpy as np
import torch
from PIL import Image, ImageOps
from IPython.display import display

ROOT = Path('/kaggle/working/f_smart_memories') if Path('/kaggle/working').exists() else Path.cwd() / 'f_smart_memories_work'
ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'python': platform.python_version(), 'torch': torch.__version__, 'device': str(DEVICE)})
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))

PROVENANCE = {'environment': {'python': platform.python_version(), 'torch': str(torch.__version__), 'device': str(DEVICE)}}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.part')
    request = urllib.request.Request(url, headers={'User-Agent': 'F-Smart-Memories-coursework'})
    try:
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open('wb') as target:
            shutil.copyfileobj(response, target)
        temporary.replace(destination)
    finally:
        if temporary.exists():
            temporary.unlink()
    return destination


## A. Tải pretrained AdaFace từ tài khoản của tác giả

Snapshot được cố định revision. File `model.safetensors` khoảng 175 MB; toàn snapshot khoảng 349 MB. Model card yêu cầu tuân theo điều kiện của dữ liệu huấn luyện.

Chỉ cài `huggingface_hub` nếu thiếu; không thay torch/CUDA của Kaggle. Không cần bật cell này để chạy phần NST.


In [ ]:
try:
    from huggingface_hub import snapshot_download
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub'], check=True)
    from huggingface_hub import snapshot_download

ADAFACE_REPO = 'minchul/cvlface_adaface_ir50_webface4m'
ADAFACE_REVISION = '60a65befbcf7e19284c4f3ac730f56867ed29594'
adaface_path = Path(snapshot_download(
    repo_id=ADAFACE_REPO,
    revision=ADAFACE_REVISION,
    local_dir=str(ROOT / 'assets' / 'adaface_ir50'),
    token=False,
))
adaface_weights = adaface_path / 'model.safetensors'
if not adaface_weights.is_file():
    raise FileNotFoundError(f'Missing checkpoint: {adaface_weights}')
PROVENANCE['adaface'] = {
    'repo': ADAFACE_REPO,
    'revision': ADAFACE_REVISION,
    'sha256': sha256_file(adaface_weights),
    'inference_run': False,
}
print({'checkpoint': str(adaface_weights), **PROVENANCE['adaface']})


## B. Lấy code Fast Neural Style từ repo PyTorch chính thức

Đầu tiên lấy commit SHA hiện tại của repo, sau đó tải source theo chính SHA đó và lưu vào manifest. Sau lần chạy thành công, nhóm nên cố định SHA trong bản dự án của mình. Cell này chỉ lấy `transformer_net.py` và license từ repo chính thức, không cần clone toàn bộ examples.


In [ ]:
api_request = urllib.request.Request(
    'https://api.github.com/repos/pytorch/examples/commits/main',
    headers={'User-Agent': 'F-Smart-Memories-coursework', 'Accept': 'application/vnd.github+json'},
)
with urllib.request.urlopen(api_request, timeout=60) as response:
    examples_revision = json.load(response)['sha']
if not re.fullmatch(r'[0-9a-f]{40}', examples_revision):
    raise ValueError('Unexpected GitHub commit SHA')

source_dir = ROOT / 'vendor' / 'pytorch_fast_neural_style'
source_dir.mkdir(parents=True, exist_ok=True)
source_url = f'https://raw.githubusercontent.com/pytorch/examples/{examples_revision}/fast_neural_style/neural_style/transformer_net.py'
transformer_source = download_file(source_url, source_dir / 'transformer_net.py')
download_file(f'https://raw.githubusercontent.com/pytorch/examples/{examples_revision}/LICENSE', source_dir / 'LICENSE')
PROVENANCE['fast_neural_style_source'] = {
    'repository': 'https://github.com/pytorch/examples',
    'revision': examples_revision,
    'source_url': source_url,
    'sha256': sha256_file(transformer_source),
}
print(PROVENANCE['fast_neural_style_source'])


## C. Tải bốn checkpoint preset

URL do [script tải của PyTorch](https://github.com/pytorch/examples/blob/main/fast_neural_style/download_saved_models.py) cung cấp. Nếu Kaggle chặn Dropbox, tải archive từ liên kết chính thức, thêm vào Kaggle Input của bạn và đặt `STYLE_ARCHIVE_INPUT` vào đường dẫn file đó. Không lấy weights từ mirror không rõ nguồn.

Đọc riêng các file tên nằm trong allowlist; không giải nén tùy ý mọi đường dẫn trong archive. Chỉ các checkpoint này được dùng trong demo.


In [ ]:
STYLE_ARCHIVE_INPUT = None  # Example: '/kaggle/input/my-model-cache/saved_models.zip'
weights_dir = ROOT / 'assets' / 'fast_neural_style'
weights_dir.mkdir(parents=True, exist_ok=True)
expected_names = {'candy.pth', 'mosaic.pth', 'rain_princess.pth', 'udnie.pth'}

if STYLE_ARCHIVE_INPUT:
    archive_path = Path(STYLE_ARCHIVE_INPUT)
else:
    archive_path = ROOT / 'assets' / 'saved_models.zip'
    if not archive_path.is_file():
        download_file('https://www.dropbox.com/s/lrvwfehqdcxoza8/saved_models.zip?dl=1', archive_path)
if not zipfile.is_zipfile(archive_path):
    raise ValueError('Download is not a ZIP. Use the official archive via Kaggle Input.')

found = set()
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        name = PurePosixPath(member.filename.replace('\\', '/')).name
        if name in expected_names and not member.is_dir():
            if name in found:
                raise ValueError(f'Duplicate checkpoint in archive: {name}')
            found.add(name)
            with archive.open(member) as source, (weights_dir / name).open('wb') as destination:
                shutil.copyfileobj(source, destination)
missing = expected_names - found
if missing:
    raise FileNotFoundError(f'Archive missing expected checkpoints: {sorted(missing)}')
PROVENANCE['fast_neural_style_weights'] = {
    'archive_sha256': sha256_file(archive_path),
    'files': {name: sha256_file(weights_dir / name) for name in sorted(expected_names)},
}
print(PROVENANCE['fast_neural_style_weights'])


## D. Chạy thử stylization bằng checkpoint có sẵn

Sửa đường dẫn ảnh ở cell dưới. Mỗi checkpoint chỉ ứng với một style; model này không nhận một ảnh style mới tùy ý. Chuyển kiểu ảnh tối đa 512 pixel ở cạnh dài để thử trước. Output giữ kích thước ảnh content đã thu nhỏ; thời gian in ra là một lần chạy thử, không phải kết quả benchmark của đồ án.

Inference nhận tensor RGB trong khoảng 0-255 theo code tham chiếu. Không áp normalization ImageNet vào input TransformerNet.


In [ ]:
CONTENT_IMAGE = Path('/kaggle/input/your-private-dataset/content.jpg')  # EDIT THIS
STYLE = 'mosaic'  # candy, mosaic, rain_princess, udnie
MAX_SIDE = 512

if not CONTENT_IMAGE.is_file():
    raise FileNotFoundError('Set CONTENT_IMAGE to an existing image in your Kaggle Input.')
if STYLE + '.pth' not in expected_names:
    raise ValueError('Choose one of the four pretrained preset styles.')

spec = importlib.util.spec_from_file_location('fsm_official_transformer_net', transformer_source)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
style_model = module.TransformerNet()
state = torch.load(weights_dir / (STYLE + '.pth'), map_location='cpu', weights_only=True)
# Historical official checkpoints contain deprecated InstanceNorm running stats.
state = {key: value for key, value in state.items() if not re.search(r'in\d+\.running_(mean|var)$', key)}
style_model.load_state_dict(state, strict=True)
style_model = style_model.to(DEVICE).eval().requires_grad_(False)

with Image.open(CONTENT_IMAGE) as source_image:
    content = ImageOps.exif_transpose(source_image).convert('RGB')
content.thumbnail((MAX_SIDE, MAX_SIDE), Image.Resampling.LANCZOS)
width, height = content.size
if min(width, height) < 8:
    raise ValueError('Image is too small for this stylization demo.')
image_tensor = torch.from_numpy(np.array(content, copy=True)).permute(2, 0, 1).float().unsqueeze(0).to(DEVICE)
# Pad to a multiple of four, then crop to the original inference dimensions.
pad_width, pad_height = (-width) % 4, (-height) % 4
image_tensor = torch.nn.functional.pad(image_tensor, (0, pad_width, 0, pad_height), mode='reflect')

with torch.inference_mode():
    style_model(image_tensor)  # warm up
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    started = time.perf_counter()
    result = style_model(image_tensor)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
result = result[0, :, :height, :width].clamp(0, 255).permute(1, 2, 0).cpu().numpy().astype(np.uint8)
output = Image.fromarray(result)
output_dir = ROOT / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'{CONTENT_IMAGE.stem}_{STYLE}.png'
output.save(output_path)
PROVENANCE['style_smoke_test'] = {'style': STYLE, 'width': width, 'height': height, 'seconds': elapsed}
print({'output': str(output_path), **PROVENANCE['style_smoke_test']})
display(content)
display(output)


## E. Lưu thông tin tái lập

Sau khi chạy thành công, download cả output và manifest để đưa vào báo cáo. Checksum ghi nhận file thực tế đã tải; checksum tự tính không thay thế việc kiểm tra nguồn phát hành.


In [ ]:
manifest_path = ROOT / 'provenance.json'
manifest_path.write_text(json.dumps(PROVENANCE, ensure_ascii=False, indent=2), encoding='utf-8')
print(manifest_path)


## Sau notebook này

- Recognition: triển khai MTCNN cho nhiều mặt, alignment 5 landmark và loader AdaFace chính chủ; chạy FaceNet baseline trên cùng split.
- NST arbitrary style: thử [StyTr2 chính thức](https://github.com/diyiiyiii/StyTR-2), hoặc [Gatys/VGG19 của PyTorch](https://docs.pytorch.org/tutorials/advanced/neural_style_tutorial.html) làm baseline.
- Không ghi nhận các con số trên benchmark của tác giả thành kết quả tự đo của nhóm.
